# Topic Modeling & NER: Overall Corpus

Runs BERTopic (already validated in `00_data_inspection_FFF.ipynb` §4) and German NER (new -- Project_Proposal.md §6.2 had no implementation yet) over the **whole** German FFF-account corpus, using the shared, reusable implementations in `src/topic_modeling.py` and `src/ner.py` rather than copy-pasted cells, so this notebook and `03_topic_model_ner_by_crisis_window.ipynb` can't silently drift apart.

**Not executed here** -- BERTopic and spaCy are both blocked locally by Smart App Control (`notes/environment_constraints.md`). Run on the GPU server.

## Setup

In [ ]:
pip install de_core_news_lg

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sshfs
import yaml 
from src import plot_style
from src.topic_modeling import prepare_topic_docs, fit_topic_model_two_pass
from src.ner import extract_entities, entity_counts_over_time

plot_style.set_style()

# data server connection
#with open("C:/Users/Elena Solar/.ssh/sshfs_config.yaml") as f:
#    config = yaml.safe_load(f)
    
with open("../../login/sshfs_config.yaml") as f:
    config = yaml.safe_load(f)

fs = sshfs.SSHFileSystem(
    host=config["sshfs"]["host"],
    username=config["sshfs"]["username"],
    client_keys=config["sshfs"]["client_keys"],
)


## Load Data

In [ ]:
with fs.open("/data/workspace/esolar/FFF_Climate_Framing/data/raw/FFF_german_01-2015_12-2022.csv") as f:
    df_1 = pd.read_csv(f, low_memory = False)

with fs.open("/data/workspace/esolar/FFF_Climate_Framing/data/raw/FFF_german_01-2023_07-2026.csv") as f:
    df_2 = pd.read_csv(f, low_memory = False)

df = pd.concat([df_1, df_2]).sort_values(by="creation_time").reset_index(drop=True)
df["creation_time"] = pd.to_datetime(df["creation_time"])
df = df[df["post_owner.username"] != "hackhermannjosef"].reset_index(drop=True)

df_FFF = pd.read_csv("../data/processed/FFF_german.csv")
df_FFF["creation_time"] = pd.to_datetime(df_FFF["creation_time"], utc=True)


print(f"{len(df_FFF)} posts from official FFF accounts")

## Topic Modeling

Two-pass fit via `src/topic_modeling.py`'s `fit_topic_model_two_pass()`, matching `00_data_inspection_FFF.ipynb` §5's validated methodology exactly (same seed list, same broad-fit params): a broad, deliberately over-aggressive fit lands around 110 over-fragmented topics first (diagnostic baseline only, not used downstream), then a refined re-fit aimed at `TARGET_N_TOPICS` is what actually gets used for everything below.

In [ ]:
seed_topic_list = [
    ["klimawandel", "erderwärmung", "co2", "emissionen"],
    ["klimakrise", "krise", "energiekrise"],
    ["corona", "covid-19"],
    ["ukraine", "krieg", "palestine", "solidarität", "dontfuelwar"],
    ["kohle", "gas", "hambi", "lützi", "kohleausstieg"],
    ["streik", "demo", "demonstration"],
    ["feminismus", "feministisch", "frauen", "gleichberechtigung", "patriarchat", "gender", "pridemonth", "bipoc"],
    ["demokratie", "wahl", "wählen", "bundestagswahl", "stimme", "rechtsextremismus", "afd", "demokratieverteidigen"],
    ["zukunft"],
    ["mobilität", "öffies", "öpnv"],
]

TARGET_N_TOPICS = 30

topic_docs, topic_dates, non_empty_mask = prepare_topic_docs(df_FFF)
print(f"{len(topic_docs)} docs ready for topic modeling ({len(df_FFF) - len(topic_docs)} empty docs excluded)")

fit_result = fit_topic_model_two_pass(topic_docs.tolist(), target_n_topics=TARGET_N_TOPICS)
topic_model, topics, topic_info = fit_result["topic_model"], fit_result["topics"], fit_result["topic_info"]

print(f"broad fit (diagnostic baseline): {fit_result['broad_n_topics']} topics, "
      f"{fit_result['broad_outlier_frac'] * 100:.1f}% outliers")
print(f"refined fit (min_cluster_size={fit_result['refined_min_cluster_size']}, target was {TARGET_N_TOPICS}): "
      f"{(topic_info['Topic'] != -1).sum()} topics, {fit_result['refined_outlier_frac'] * 100:.1f}% outliers")
topic_info.head(20)

In [ ]:
all_topics = topic_info[topic_info["Topic"] != -1].sort_values("Count", ascending=False)

fig, ax = plt.subplots(figsize=(plot_style.FIGSIZE_TALL[0], max(plot_style.FIGSIZE_TALL[1], 0.28 * len(all_topics))))
sns.barplot(data=all_topics, y="Name", x="Count", order=all_topics["Name"], ax=ax)
plot_style.annotate_bars(ax)
ax.set_title(f"BERTopic Topics by Size -- Overall Corpus ({len(all_topics)} topics)")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Topic")
#plot_style.savefig(fig, "FFF/overall_bertopic_topics")
plt.show()

### Topics Over Time

Two views: a heatmap covering every topic (scales far better to ~30 topics than an overlapping line plot would), plus a focused line plot for the handful of biggest topics where individual trajectories are worth reading in detail.

In [ ]:
topics_over_time = topic_model.topics_over_time(topic_docs.tolist(), topic_dates.tolist(), nr_bins=40)
topic_names = topic_info.set_index("Topic")["Name"]

heatmap_data = (
    topics_over_time[topics_over_time["Topic"] != -1]
    .pivot(index="Topic", columns="Timestamp", values="Frequency")
    .fillna(0)
)
heatmap_data = heatmap_data.loc[heatmap_data.sum(axis=1).sort_values(ascending=False).index]
heatmap_data.index = heatmap_data.index.map(topic_names)
heatmap_data.columns = heatmap_data.columns.strftime("%Y-%m")

fig, ax = plt.subplots(figsize=(10, max(6, 0.3 * len(heatmap_data))))
sns.heatmap(heatmap_data, cmap=plot_style.PALETTE_SEQUENTIAL, ax=ax, xticklabels=4, cbar_kws={"label": "Posts per time bin"})
ax.set_title("Topic Frequency Over Time -- All Topics")
ax.set_xlabel("Date")
ax.set_ylabel("Topic")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
#plot_style.savefig(fig, "FFF/overall_topics_heatmap")
plt.show()

In [ ]:
top_topic_ids = topic_info[topic_info["Topic"] != -1].sort_values("Count", ascending=False)["Topic"].head(8).tolist()
plot_data = topics_over_time[topics_over_time["Topic"].isin(top_topic_ids)].copy()
plot_data["Topic Name"] = plot_data["Topic"].map(topic_names)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.lineplot(data=plot_data, x="Timestamp", y="Frequency", hue="Topic Name", ax=ax)
ax.set_title("Top 8 Topics Over Time -- Detail View")
ax.set_xlabel("Date")
ax.set_ylabel("Posts per Time Bin")
ax.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
#plot_style.savefig(fig, "FFF/overall_topics_over_time_top8")
plt.show()

## Named Entity Recognition



In [ ]:
ner_docs = df_FFF["text"]
ner_ids = df_FFF["id"].astype(str) if "id" in df_FFF.columns else df_FFF.index.astype(str)

entities = extract_entities(ner_docs.tolist(), ner_ids.tolist())
print(f"{len(entities)} entity mentions across {entities['text_id'].nunique()} posts")
entities["entity_text"].value_counts().head(20)

In [ ]:
dates_lookup = pd.DataFrame({"text_id": ner_ids, "creation_time": df_FFF["creation_time"]})
entity_timeline = entity_counts_over_time(entities, dates_lookup, top_n=8)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
entity_timeline.plot(ax=ax, linewidth=1.2)
ax.set_title("Top Named Entities Over Time -- Overall Corpus")
ax.set_xlabel("Month")
ax.set_ylabel("Mentions")
ax.legend(title="Entity", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plot_style.savefig(fig, "overall_entity_timeline")
plt.show()

## Save Results

In [ ]:
topic_names = topic_info.set_index("Topic")["Name"]

# non_empty_mask comes straight from prepare_topic_docs -- it is aligned to df_FFF's
# original index, which is what topics (from fit_topic_model_two_pass) is ordered by. Do NOT
# recompute an "is text non-empty" mask independently here: a post can have non-empty
# RAW text but empty CLEANED text (e.g. a hashtags-only post), so the two masks disagree
# and recomputing one caused a real "Length of values (4104) != length of index (4138)" bug.
df_topics = df_FFF.loc[non_empty_mask].copy()
df_topics["topic"] = topics
df_topics["topic_name"] = df_topics["topic"].map(topic_names)

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
df_topics.to_csv(output_dir / "overall_topics.csv", index=False)
entities.to_csv(output_dir / "overall_entities.csv", index=False)
print(f"saved {len(df_topics)} topic assignments and {len(entities)} entity mentions to {output_dir}")